# Why FLARE works

### Understanding the design of a transient classifier

Most tutorials show you *how* to run a model. This one asks a harder question:
**why is this the right model for this problem?**

FLARE classifies ZTF light curves into the six families of the paper: Type Ia
supernovae (SN Ia), core-collapse supernovae (SN CC), superluminous supernovae
(SLSN), active galactic nuclei (AGN), tidal disruption events (TDE) and
cataclysmic variables (CV). It does this with physically motivated features,
catalogue context and gradient-boosted trees, and without a deep sequence model.
That choice was made by measurement, and in this notebook we repeat the
measurements.

By the end we will have re-derived, on our own machine, the six findings the
design rests on:

| # | Claim | We test it by |
|---|---|---|
| 1 | The data regime decides the method | counting objects and rare events |
| 2 | The features carry the signal | training on 5 → 190 columns |
| 3 | The model family matters much less | four different learners, same features |
| 4 | The features are readable physics | reading what separates a TDE from an AGN |
| 5 | Calibration comes from *how* we fit, more than *what* we fit | reliability diagrams |
| 6 | Real uncertainty needs conformal | coverage of thresholds vs. prediction sets |

Everything uses the BTS benchmark that ships in `benchmark/` and runs on a CPU in
a few minutes. Every model in the comparisons is trained on the training split,
early-stopped on the validation split and scored on the test split. For the
didactic comparisons we use a flat six-class LightGBM model; the shipped FLARE
classifier is hierarchical (a five-class top model in which SN CC and SLSN share
one node, followed by a SLSN branch) and is trained on train and validation
together, so we quote it only as a reference on the test split.

In [ ]:
import sys, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, f1_score, recall_score)

sys.path.insert(0, "..")
import flare
from flare import benchmark as B
from flare import data as D
from flare.features import extract_from_array
from flare.metrics import ece

B.prepare()                       # unpacks benchmark/light_curves.tar.gz once
CLASSES = B.SIX                   # ["SN_Ia", "SN_CC", "SLSN", "AGN", "TDE", "CV"]
K = len(CLASSES)
LABEL = {"SN_Ia": "SN Ia", "SN_CC": "SN CC", "SLSN": "SLSN",
         "AGN": "AGN", "TDE": "TDE", "CV": "CV"}

# one fixed colour per class, six evenly spaced magma shades (the near-white end is left out)
PALETTE = [tuple(c) for c in plt.cm.magma(np.linspace(0.08, 0.82, K))]
CLASS_COLOR = dict(zip(CLASSES, PALETTE))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
RNG = 42

def metrics6(y, p):
    """Six-class scores: accuracy, balanced accuracy, macro F1, macro AUPRC, ECE, recalls."""
    pred = p.argmax(1)
    out = {"accuracy": accuracy_score(y, pred),
           "balanced_accuracy": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=list(range(K))),
           "macro_auprc": np.mean([average_precision_score(y == c, p[:, c])
                                   for c in range(K)]),
           "ece": ece(y, p)}
    rec = recall_score(y, pred, average=None, labels=list(range(K)), zero_division=0)
    out.update({f"recall_{c}": r for c, r in zip(CLASSES, rec)})
    return out

print("Setup OK")

---
## 1. The data regime decides the method

Before choosing a model, we count the data. This single number does more to
determine what will work than any architectural preference.

In [ ]:
X_train, y_train, t_train = B.load_split("train")
X_val,   y_val,   t_val   = B.load_split("val")
X_test,  y_test,  t_test  = B.load_split("test")

LC_COLS = [c for c in pd.read_parquet(B.BENCHMARK / "features_all.parquet").columns
           if c != "obj_id"]
CTX_COLS = [c for c in X_train.columns if c not in LC_COLS]

rows = []
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    rows.append({"split": name, "objects": len(y),
                 **{LABEL[c]: int((y == i).sum()) for i, c in enumerate(CLASSES)}})
counts = pd.DataFrame(rows).set_index("split")
print(counts.to_string())
print(f"\nColumns per object: {X_train.shape[1]} "
      f"({len(LC_COLS)} light-curve features + {len(CTX_COLS)} context columns)")
for c in ["TDE", "SLSN"]:
    n = int((y_train == CLASSES.index(c)).sum())
    print(f"{LABEL[c]:4s} in training: {n:4d} examples ({100 * n / len(y_train):.2f}% of the training set)")

Each of those objects is a multi-band light curve. Here is the best-sampled TDE in
the training split, and a check that the 160 light-curve features stored in the
benchmark are exactly what `flare.features` computes from it.

In [ ]:
# what one of those rare objects looks like: the best-sampled TDE in the training split
i_tde = t_train[y_train == CLASSES.index("TDE")].n_events.idxmax()
row = t_train.loc[i_tde]
arr = D.load_events(str(B.BENCHMARK / row.filepath))
bands = D.reconstruct_bands(arr)

BAND_COLOR = {"g": PALETTE[1], "r": PALETTE[3], "i": PALETTE[5]}
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for b, lc in bands.items():
    ax.errorbar(lc.t, lc.flux, yerr=lc.flux_err, fmt="o", ms=4, capsize=0,
                color=BAND_COLOR[b], label=f"ZTF {b} ({len(lc.t)} points)")
ax.set_xlabel("days since first detection"); ax.set_ylabel(r"flux ($\mu$Jy)")
ax.set_title(f"{row.obj_id}, a {row.bts_type}")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

# the benchmark's feature table is exactly what flare.features computes from this array
f = pd.Series(extract_from_array(arr))
same = np.allclose(f.values.astype(float), X_train.loc[i_tde, f.index].values.astype(float),
                   rtol=1e-5, equal_nan=True)
print(f"{len(f)} features recomputed from the light curve; identical to the benchmark table: {same}")
n_ev = pd.concat([t_train, t_val, t_test]).n_events
print(f"points per light curve over the benchmark: median {n_ev.median():.0f}, "
      f"5th to 95th percentile {n_ev.quantile(0.05):.0f} to {n_ev.quantile(0.95):.0f}")

Those numbers are the whole argument.

There are **7,347** training objects, and 5,011 of them are SN Ia. There are
**47** TDEs and **109** SLSNe, the classes astronomers most want to find, each
below 1.5% of the training set. Each object is an irregularly sampled,
heteroscedastic time series in two or three filters, with a median of 25 points
(10 to 73 between the 5th and 95th percentiles), gaps of weeks and a different
cadence in every band. The TDE above is one of the best-sampled objects in the
benchmark, with 149 points, and it still has a gap of about ten days.

Deep sequence models learn their own representation from raw data. That is a
strength when there are $10^5$ to $10^7$ examples to learn it from. With $10^4$
objects and 47 examples of the rarest class, a learned representation is mostly
learning the noise of this particular sample. A century of astronomy already
tells us what matters in a light curve: how fast it rises, how slowly it fades,
what colour it is, whether the colour changes, and where on its host galaxy it
sits.

**In the small-data regime, injecting known physics works better than learning it
from scratch.** The rest of this notebook is the evidence.

---
## 2. The features carry the signal

FLARE computes 160 numbers from each light curve and joins 30 columns of
catalogue context that exist when the alert is issued: Pan-STARRS host
photometry and offset, a pseudo absolute magnitude `M_pseudo` from the Legacy
Surveys photometric redshift, Gaia and AllWISE matches, and the Pan-STARRS
counterpart at the position. Do we need all 190 columns? And how much does
performance depend on having good ones?

We train the same model on the best 5 columns, then 10, 20, 40, 80, 120 and all
190, ranking them by how much a full model actually uses them (total gain). We
also train one model on the 160 light-curve features alone.

In [ ]:
import lightgbm as lgb

PARAMS = dict(objective="multiclass", num_class=K, metric="multi_logloss",
              verbosity=-1, learning_rate=0.05, num_leaves=31, min_data_in_leaf=10,
              feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
              num_threads=4, seed=RNG, deterministic=True, force_row_wise=True)

def fit(X_tr, y_tr, X_va, y_va, params=PARAMS, weight=None, rounds=2000):
    """Train on the training split, stop when the validation log-loss stops improving."""
    dtr = lgb.Dataset(X_tr, label=y_tr, weight=weight)
    dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
    return lgb.train(params, dtr, num_boost_round=rounds, valid_sets=[dva],
                     callbacks=[lgb.early_stopping(50, verbose=False)])

def score(m, X, y):
    return metrics6(y, m.predict(X, num_iteration=m.best_iteration))

# rank all 190 columns by total gain in a full model
full = fit(X_train, y_train, X_val, y_val)
gain = pd.Series(full.feature_importance("gain"),
                 index=X_train.columns).sort_values(ascending=False)

ks, curve = [5, 10, 20, 40, 80, 120, len(X_train.columns)], []
for k in ks:
    if k == len(gain):                       # all columns: reuse the full model, original order
        cols, m = list(X_train.columns), full
    else:
        cols = list(gain.index[:k])
        m = fit(X_train[cols], y_train, X_val[cols], y_val)
    met = score(m, X_test[cols], y_test)
    curve.append({"k": k, **met})
    print(f"  top {k:3d} columns: acc={met['accuracy']:.3f}  bal.acc={met['balanced_accuracy']:.3f}  "
          f"F1={met['macro_f1']:.3f}  AUPRC={met['macro_auprc']:.3f}")
curve = pd.DataFrame(curve)

# the same model without the catalogue context: light-curve features only
lc_only = fit(X_train[LC_COLS], y_train, X_val[LC_COLS], y_val)
met_lc = score(lc_only, X_test[LC_COLS], y_test)
print(f"\n  {len(LC_COLS)} light-curve features only: acc={met_lc['accuracy']:.3f}  "
      f"bal.acc={met_lc['balanced_accuracy']:.3f}  F1={met_lc['macro_f1']:.3f}  "
      f"AUPRC={met_lc['macro_auprc']:.3f}")

In [ ]:
METRIC_STYLE = [("accuracy", "accuracy", PALETTE[0]),
                ("balanced_accuracy", "balanced accuracy", PALETTE[2]),
                ("macro_f1", "macro $F_1$", PALETTE[3]),
                ("macro_auprc", "macro AUPRC", PALETTE[5])]
fig, ax = plt.subplots(figsize=(6.4, 3.9))
for col, lab, colr in METRIC_STYLE:
    ax.plot(curve.k, curve[col], "o-", color=colr, label=lab)
    ax.plot(len(LC_COLS), met_lc[col], "x", ms=9, mew=2, color=colr)
ax.plot([], [], "kx", ms=8, mew=2, label="light curve only (160)")
ax.set_xscale("log"); ax.set_xticks(ks); ax.set_xticklabels(ks); ax.minorticks_off()
ax.set_xlabel("number of columns used (ranked by gain)"); ax.set_ylabel("test score")
ax.set_title("Performance saturates once the physics is in")
ax.legend(frameon=False, fontsize=8.5, loc="lower right")
plt.tight_layout(); plt.show()

a, b = curve.iloc[0], curve.iloc[-1]
print(f"5 -> {int(b.k)} columns:  accuracy {a.accuracy:.3f} -> {b.accuracy:.3f}   "
      f"balanced accuracy {a.balanced_accuracy:.3f} -> {b.balanced_accuracy:.3f}   "
      f"macro F1 {a.macro_f1:.3f} -> {b.macro_f1:.3f}")

Three things stand out.

**The curve rises steeply and then flattens.** Going from 5 to 20 columns lifts
accuracy from 0.855 to 0.907 and macro $F_1$ from 0.545 to 0.786; beyond 40
columns accuracy stays between 0.917 and 0.922. That is what a *sufficient*
representation looks like: once the physics is described, adding more
description changes little.

**The gain from a poor representation to a good one is large.** Between 5 columns
and all 190, accuracy rises by 6.4 points (0.855 to 0.919), balanced accuracy by
20.5 points (0.505 to 0.710) and macro $F_1$ by 23.2 points (0.545 to 0.777).
The context block is part of that representation: the 160 light-curve features
alone reach an accuracy of 0.887, a balanced accuracy of 0.575 and a macro $F_1$
of 0.644 (the crosses), so the catalogue columns add 13.5 points of balanced
accuracy. We keep these magnitudes in mind for §3.

**Balanced accuracy and macro $F_1$ wobble once the curve has flattened.** Both
average over classes, so the rare classes weigh as much as SN Ia, and this test
split holds only 10 TDEs and 24 SLSNe. One TDE changing its label moves balanced
accuracy by $1/(6 \times 10)$, about 1.7 points, comparable to the 2.1-point
dip between 20 and 40 columns. Accuracy and AUPRC, which lean on the well-populated
classes, are much steadier. This is the first warning that **rare-class numbers
from one split are noisy**, a theme we return to at the end.

Next we look at the top of the ranking to see which columns the model leans on.

In [ ]:
top = gain.head(15)[::-1]
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.barh(range(len(top)), top.values / top.values.max(),
        color=[PALETTE[1] if f in LC_COLS else PALETTE[4] for f in top.index])
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=9)
from matplotlib.patches import Patch
ax.set_xlabel("relative importance (gain)")
ax.set_title("The 15 columns the model relies on most")
ax.legend(handles=[Patch(color=PALETTE[1], label="light-curve feature"),
                   Patch(color=PALETTE[4], label="catalogue context")],
          frameon=False, loc="lower right")
plt.tight_layout(); plt.show()
print("context columns among the top 15:", [f for f in gain.index[:15] if f in CTX_COLS])

The three columns at the top describe how steadily the light curve evolves: the
uncertainty of the linear-trend slope in $g$ and the linear slope and trend in
$r$, which separate smooth explosive transients from stochastic variables. Peak times, the $g-r$
colour and the rise time follow. Five of the top fifteen are catalogue context:
the Gaia proper motion `pm` (which flags Galactic stars, hence CVs),
`M_pseudo`, the host colour and extendedness, and the WISE $W1-W2$ colour, which
is the classic mid-infrared AGN indicator.

---
## 3. The model family matters far less than the features

If the features really are doing the work, then *what consumes them* should
matter much less than people assume. We test that directly by giving the same
190 columns to four very different learners. The scikit-learn models cannot read
missing values, so they get the training median in their place; LightGBM handles
missing values natively.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

Xtr = X_train.to_numpy(dtype=float); Xte = X_test.to_numpy(dtype=float)

def sk(model):
    # the scikit-learn models cannot read NaN, so missing values get the training median
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), model)

learners = {
    "logistic regression": sk(LogisticRegression(max_iter=3000, C=1.0)),
    "random forest":       sk(RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                                     random_state=RNG, n_jobs=4)),
    "neural network (MLP)": sk(MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=400,
                                             random_state=RNG, early_stopping=True)),
}

def row(name, p, fit_s):
    m = metrics6(y_test, p)
    return {"model": name, "accuracy": m["accuracy"], "balanced acc": m["balanced_accuracy"],
            "macro F1": m["macro_f1"], "macro AUPRC": m["macro_auprc"], "ECE": m["ece"],
            "fit (s)": fit_s}

rows, probs = [], {}
for name, mdl in learners.items():
    t0 = time.time(); mdl.fit(Xtr, y_train); dt = time.time() - t0
    probs[name] = mdl.predict_proba(Xte)
    rows.append(row(name, probs[name], dt))
    print(f"  {name:22s} fitted in {dt:5.1f}s")

probs["gradient boosting"] = p_lgb = full.predict(X_test, num_iteration=full.best_iteration)
rows.append(row("gradient boosting", p_lgb, np.nan))
comp = pd.DataFrame(rows).set_index("model")
print()
print(comp.round(3).to_string())

# for reference: the shipped classifier (hierarchical, trained on train+val)
clf = flare.load_classifier()
p_ship = clf.predict_proba(X_test.reindex(columns=clf.feature_names))
ms = metrics6(y_test, p_ship)
print(f"\nshipped FLARE (hierarchical, train+val): acc={ms['accuracy']:.3f}  "
      f"bal.acc={ms['balanced_accuracy']:.3f}  F1={ms['macro_f1']:.3f}  "
      f"AUPRC={ms['macro_auprc']:.3f}  ECE={ms['ece']:.3f}")

In the **accuracy** column, four learners with very different inductive biases (a
linear model, a bagged forest, a neural network and a boosted ensemble) land
between 0.879 and 0.919, a spread of 4.0 points, and their macro $F_1$ values lie
between 0.737 and 0.777. In §2, changing the *representation* moved accuracy by
6.4 points and macro $F_1$ by 23.2 points.

That ratio is the point. **Once the representation is right, the choice of
classifier is a second-order decision.** It is also why FLARE does not use a deep
sequence model: a deep model's job is to *learn* the representation, and here the
representation is already known and cheap to compute.

Second-order is still not zero. Boosting has the best accuracy (0.919), macro
$F_1$ (0.777) and macro AUPRC (0.887). It does not win everything: logistic
regression has the highest balanced accuracy (0.737 against 0.710), so on the
rare classes the ranking is not settled by this table. §7 shows that a class
weight moves the boosted model well past that.

The shipped classifier reaches 0.928 accuracy, 0.844 balanced accuracy and 0.851
macro $F_1$ on the same test split. It is not a fair entrant in this table,
since it is trained on train and validation together, has its own SLSN branch and
was trained with the context blocks randomly removed, but it shows where the
complete recipe lands.

Now we look at the **ECE** column, the expected calibration error. Here the
models are *not* interchangeable: it ranges from 0.017 for boosting to 0.080 for
the random forest, almost a factor of five, and it does not track accuracy. The
random forest is the most accurate of the three scikit-learn models and the worst
calibrated of all four. Accuracy alone would never have told us that. The next
section looks at why.

---
## 4. The features are readable physics

A representation we designed is a representation we can interrogate. When we ask
the model why it calls something a TDE, the answer is a sentence about astronomy
rather than a heat map over a hidden layer. We compute SHAP values of the boosted
model on 600 test objects (all test TDEs, AGN and SLSNe, plus a random draw of
the rest) and rank the columns by their mean absolute contribution to the TDE
output.

In [ ]:
import shap

expl = shap.TreeExplainer(full)
# all test TDEs, AGN and SLSNe plus a random draw of the rest, 600 objects in total
rare = X_test.index[np.isin(y_test, [CLASSES.index(c) for c in ["TDE", "AGN", "SLSN"]])]
rest = X_test.drop(index=rare).sample(600 - len(rare), random_state=RNG).index
sub = X_test.loc[rare.append(rest)]
sv = expl.shap_values(sub)
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)   # (n, feat, class)

tde_idx = CLASSES.index("TDE")
tde_shap = pd.Series(np.abs(sv[:, :, tde_idx]).mean(axis=0),
                     index=sub.columns).sort_values(ascending=False)
print("What drives the TDE decision (mean |SHAP| on the TDE output):\n")
for name, val in tde_shap.head(8).items():
    kind = "light curve" if name in LC_COLS else "context"
    print(f"  {name:28s} {val:.4f}   [{kind}]")

In [ ]:
# the two most important TDE features that most objects have, TDE against its main confusers
feats = [f for f in tde_shap.head(12).index if X_test[f].notna().mean() > 0.6][:2]
fig, axes = plt.subplots(1, len(feats), figsize=(5.2 * len(feats), 3.6))
axes = np.atleast_1d(axes)
X_all = pd.concat([X_train, X_val, X_test]); y_all = np.concatenate([y_train, y_val, y_test])
for ax, f in zip(axes, feats):
    lo, hi = np.nanpercentile(X_all[f].dropna(), [1, 99])
    for cls in ["SN_Ia", "AGN", "TDE"]:
        v = X_all[f][y_all == CLASSES.index(cls)].dropna()
        ax.hist(np.clip(v, lo, hi), bins=30, range=(lo, hi), density=True, alpha=0.55,
                color=CLASS_COLOR[cls], label=f"{LABEL[cls]} ({len(v)})")
    ax.set_xlabel(f); ax.set_ylabel("density"); ax.legend(frameon=False)
plt.suptitle("Why the model can tell a TDE from an AGN and a SN Ia (all three splits)", y=1.02)
plt.tight_layout(); plt.show()
for f in feats:
    med = {LABEL[c]: X_all[f][y_all == CLASSES.index(c)].median() for c in ["SN_Ia", "AGN", "TDE"]}
    print(f"median {f}: " + ",  ".join(f"{k} {v:.3g}" for k, v in med.items()))

Two kinds of evidence come out on top, and the histograms (all three splits
together, so that the TDE histogram holds 68 objects) show why.

**Where the transient sits.** The most important column is `host_sep`, the
offset from the Pan-STARRS host in arcseconds. TDEs happen in the nucleus of
their galaxy: their median offset is 0.123 arcsec, against 2.93 arcsec for SN
Ia. This separates TDEs from supernovae, but AGN are nuclear too (median 0.144
arcsec), so position alone cannot split a TDE from an AGN.

**What colour it is, and whether the colour changes.** The next three columns are
the mean $g-r$ colour, its slope and its late-time value. **Tidal disruption
events are blue and they stay blue**, because the accretion flow holds a roughly
constant temperature for months: the median mean $g-r$ is $-0.167$ mag for TDEs,
against $+0.216$ for AGN and $+0.275$ for SN Ia. Supernovae start blue and redden
as they expand and cool; AGN vary stochastically without a coherent colour trend.

That is textbook transient physics, recovered by the model from data. It is also
*auditable*: an astronomer can look at this and agree or disagree, which is a
different relationship with a model than trusting it.

---
## 5. Honest probabilities

A classifier that says "90% confident" should be right 90% of the time it says
that. Many are not, and it matters, because in a real survey the probability is
what decides whether a telescope slews.

We plot predicted confidence against observed accuracy. The diagonal is perfect
calibration.

In [ ]:
def reliability(y, p, n_bins=12):
    conf, pred = p.max(1), p.argmax(1)
    correct = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 8:
            xs.append(conf[m].mean()); ys.append(correct[m].mean()); ns.append(m.sum())
    return np.array(xs), np.array(ys), np.array(ns)

fig, ax = plt.subplots(figsize=(5.4, 4.6))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
for name, lab, colr in [("gradient boosting", "gradient boosting", PALETTE[0]),
                        ("neural network (MLP)", "neural network", PALETTE[2]),
                        ("logistic regression", "logistic regression", PALETTE[3]),
                        ("random forest", "random forest", PALETTE[4])]:
    xs, ys, _ = reliability(y_test, probs[name])
    ax.plot(xs, ys, "o-", color=colr, label=f"{lab}  (ECE {ece(y_test, probs[name]):.3f})")
ax.set_xlabel("predicted confidence"); ax.set_ylabel("observed accuracy")
ax.set_title("Does 90% confident mean 90% right?")
ax.legend(frameon=False, loc="upper left", fontsize=8.5)
plt.tight_layout(); plt.show()

for name in ["gradient boosting", "random forest"]:
    p = probs[name]; hi_ = p.max(1) >= 0.9
    print(f"{name:18s}: {hi_.mean():.0%} of test objects at confidence >= 0.9, "
          f"mean confidence there {p.max(1)[hi_].mean():.3f}, "
          f"accuracy there {(p.argmax(1)[hi_] == y_test[hi_]).mean():.3f}")

The boosted model tracks the diagonal closely (ECE 0.017): of the 78% of test
objects it scores at 0.9 or above, it gets 97.4% right at a mean confidence of
97.7%. The neural network and logistic regression fall mostly below the
diagonal at intermediate confidence, so there they are overconfident.

The **random forest sits above the diagonal** everywhere: it is
*underconfident*. Only 42% of the test objects reach a confidence of 0.9, and
among those it is right 99.1% of the time while claiming 95.9%. Its probability
is the fraction of 400 bagged trees voting for a class, and trees grown on
different bootstrap samples and feature subsets disagree often enough, across six
classes, to pull that fraction away from 0 and 1. An underconfident model is
safer than an overconfident one, but it still misreports its own reliability,
and it is the most accurate of the three scikit-learn models.

So the folklore ("neural networks are overconfident, trees are not") does not
describe this experiment, and neither does "trees are well calibrated": the two
tree ensembles sit at the best and the worst ECE in the table.

**What matters most is how the model is fitted.** The booster minimises the
multi-class log-loss, a proper scoring rule, and stops the moment the *validation*
log-loss stops improving, so it never gets to sharpen its probabilities beyond
what held-out data supports. The MLP also stops early, but scikit-learn's early
stopping monitors validation *accuracy*, which says nothing about the
probabilities. Logistic regression minimises log-loss on the training data only,
and the random forest optimises no probability at all.

Calibration is therefore not only a post-processing step bolted on at the end. It
falls out of the fitting procedure, and we get it by **optimising a proper
scoring rule on held-out data and stopping when it stops improving**. That is one
of the reasons FLARE's classifier is an early-stopped boosted ensemble, and it is
a property we must *measure* for each model rather than assume from its family.

---
## 6. Uncertainty we can actually promise

Good calibration is not a guarantee. The usual way to be careful is a confidence
threshold: "only act when the model is more than 90% sure." It sounds safe. We
measure what it does for each class.

The alternative is **conformal prediction**, which returns a *set* of classes
instead of one label and comes with a coverage guarantee: the true class is in
the set at least 90% of the time. Calibrated separately for each class (a
*Mondrian* conformal predictor), that promise is made for the rare classes too.
We calibrate on the validation split here; the shipped model is calibrated
cross-conformally over five folds.

In [ ]:
def mondrian_calibrate(p_cal, y_cal, alpha):
    """Per-class conformal quantile of the score s = 1 - p(true class)."""
    q = np.ones(K)
    for c in range(K):
        s = np.sort(1.0 - p_cal[y_cal == c, c]); n = len(s)
        k = int(np.ceil((n + 1) * (1 - alpha)))
        q[c] = 1.0 if k > n else s[k - 1]
    return q

def prediction_sets(p, qhat):
    return p >= (1.0 - qhat)[None, :]          # boolean (N, K) membership

# the validation split already early-stopped the booster, so it is not a clean
# calibration set; we use it here for simplicity and return to this point below
p_val = full.predict(X_val, num_iteration=full.best_iteration)
alpha = 0.10
qhat = mondrian_calibrate(p_val, y_val, alpha)
sets = prediction_sets(p_lgb, qhat)

THRESH = 0.90
conf, pred = p_lgb.max(1), p_lgb.argmax(1)
rows = []
for i, cls in enumerate(CLASSES):
    m = y_test == i
    kept = conf[m] >= THRESH
    rows.append({
        "class": LABEL[cls], "n (test)": int(m.sum()), "n (cal)": int((y_val == i).sum()),
        "threshold: kept": f"{kept.mean():.0%}",
        "threshold: correct among kept":
            f"{(pred[m][kept] == i).mean():.0%}" if kept.sum() else "none kept",
        "conformal: coverage": f"{sets[m, i].mean():.0%}",
        "conformal: avg set size": f"{sets[m].sum(1).mean():.2f}",
    })
print(f"Confidence threshold = {THRESH:.0%};  conformal target = {1-alpha:.0%}\n")
print(pd.DataFrame(rows).set_index("class").to_string())
print(f"\nmarginal coverage {sets[np.arange(len(y_test)), y_test].mean():.1%}, "
      f"mean set size {sets.sum(1).mean():.2f}, "
      f"single-class sets {(sets.sum(1) == 1).mean():.0%}")

In [ ]:
cov = [sets[y_test == i, i].mean() for i in range(K)]
kept = [(conf[y_test == i] >= THRESH).mean() for i in range(K)]
labs = [LABEL[c] for c in CLASSES]

x = np.arange(K); w = 0.38
fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.bar(x - w/2, kept, w, label=f"kept by a {THRESH:.0%} threshold", color=PALETTE[4])
ax.bar(x + w/2, cov, w, label="conformal coverage", color=PALETTE[1])
ax.axhline(1 - alpha, ls="--", c="k", lw=1, label=f"{1-alpha:.0%} target")
ax.set_xticks(x); ax.set_xticklabels(labs); ax.set_ylim(0, 1.05)
ax.set_ylabel("fraction of test objects")
ax.set_title("A threshold quietly discards the rare classes")
ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
plt.tight_layout(); plt.show()

# the shipped classifier is calibrated cross-conformally over the paper's five folds
print("shipped model, out-of-sample per-class coverage at 90% (five-fold, from the paper):")
print("  " + ",  ".join(f"{LABEL[c]} {v:.1%}" for c, v in clf.coverage.items()))

The confidence threshold is not neutral. It keeps 85% of the SN Ia and 82% of
the AGN, but only 38% of the SLSNe and 40% of the TDEs, the rare objects the
classifier exists to find. It does this *silently*: the reported accuracy goes
up, because the hard cases were deleted. Even the survivors are unreliable for
the rarest classes: only 44% of the SLSNe kept by the threshold and 75% of the
TDEs (three of four) are classified correctly.

Conformal prediction makes the opposite trade. Nothing is discarded. When the
model is unsure it returns two or three candidate classes instead of one, and the
average set size says how often that happens: 1.05 for SN Ia, 1.75 for SLSNe and
2.80 for TDEs. Over the whole test split the sets cover the true class 89.7% of
the time with a mean size of 1.15, and 83% of them hold a single class. A
shortlist of two classes we can trust is far more useful to a follow-up programme
than a single label we cannot.

**Now the honest caveats, which are themselves the lesson.**

The guarantee is a *finite-sample* one, and here it rests on 11 validation TDEs
and 23 SLSNe. With $n$ calibration objects the attainable levels step in units
of $1/(n{+}1)$, and for TDEs at 90% the rule $\lceil (n+1)(1-\alpha) \rceil = 11$
picks the *largest* of the 11 calibration scores. The TDE threshold is therefore
set by the single least TDE-like TDE in the validation split, which is why the
TDE sets are so large. The realised test coverage of 90% is 9 of 10 objects and
would move by ten points with one object.

SN Ia, with over a thousand objects, falls slightly short at 88%. Part of the
reason is that the validation split also early-stopped the booster, so its scores
are slightly optimistic and not perfectly exchangeable with the test split. A
clean calibration set is one that nothing else has touched.

The fix is more and cleaner calibration data, not a different method. The
shipped classifier is calibrated cross-conformally over the paper's five folds,
and its out-of-sample coverage at 90% lies between 89.2% (AGN) and 91.4% (TDE)
for every class. The rule to take away is that **a guarantee conditioned on a
class we have eleven examples of is a weak guarantee**, and a method that lets us
see this is better than one that hides it.

---
## 7. One knob for the rare-class trade-off

We will always be asked: *can it find more TDEs?* Yes, and the cost is explicit.
We weight the training examples by $(1/n_c)^\beta$, where $n_c$ is the size of
class $c$. $\beta = 0$ ignores the imbalance; larger $\beta$ pushes the model
toward the rare classes.

In [ ]:
def weights(y, beta):
    counts = np.bincount(y, minlength=K).astype(float)
    return ((len(y) / (K * counts)) ** beta)[y]

betas, knob = [0.0, 0.5, 0.9, 1.25, 1.75], []
for b in betas:
    m = fit(X_train, y_train, X_val, y_val, weight=weights(y_train, b))
    met = score(m, X_test, y_test)
    knob.append({"beta": b, "accuracy": met["accuracy"],
                 "balanced acc": met["balanced_accuracy"], "macro F1": met["macro_f1"],
                 "TDE recall": met["recall_TDE"], "SLSN recall": met["recall_SLSN"],
                 "SN Ia recall": met["recall_SN_Ia"]})
knob = pd.DataFrame(knob).set_index("beta")
print(knob.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.9))
# solid lines: averages over all classes; dashed: recall of one rare class, in its class colour
for col, colr, ls in [("accuracy", PALETTE[0], "-"), ("balanced acc", PALETTE[1], "-"),
                      ("macro F1", PALETTE[5], "-"), ("SLSN recall", CLASS_COLOR["SLSN"], "--"),
                      ("TDE recall", CLASS_COLOR["TDE"], "--")]:
    ax.plot(knob.index, knob[col], "o", ls=ls, color=colr, label=col)
ax.set_xlabel(r"class-weight strength $\beta$"); ax.set_ylabel("test score")
ax.set_title("Buying rare-class recall, and what it costs")
ax.legend(frameon=False, fontsize=8.5)
plt.tight_layout(); plt.show()

The curves are more interesting than the usual slogan.

Balanced accuracy climbs from 0.710 at $\beta = 0$ to 0.871 at $\beta = 1.75$.
TDE recall goes from 0.4 to 0.9 (from 4 to 9 of the 10 test TDEs) by
$\beta = 0.9$, and SLSN recall from 0.333 to 0.708 (from 8 to 17 of 24). Overall
accuracy, meanwhile, barely moves: it *rises* from 0.919 to 0.931 at moderate
weights and only falls back to 0.910 at $\beta = 1.75$. Over the range
$0.5 \le \beta \le 1.25$ the trade-off is close to *free* on this split.

Accuracy is dominated by SN Ia, which are 68% of the test split. Reweighting moves
a modest number of borderline objects out of that class and into the rare ones:
SN Ia recall falls from 0.972 to 0.936 at the strongest weight. We lose a little
where we have thousands and gain a lot where we have tens, and accuracy, being an
average over objects and not over classes, hardly notices.

Two lessons follow. First, **when the classes are this imbalanced, accuracy is a
poor model-selection criterion**; it stays within about two points across models
whose rare-class recall differs by a factor of two. Second, the cost of class
weighting is real but it shows up in *other* places: the recall and purity of the
common classes, and macro $F_1$, which stops improving after $\beta = 0.5$ while
balanced accuracy keeps rising. Those are what we must watch.

Which point we want is a *scientific* decision, not a machine-learning one. A
broker hunting TDEs and a population study of SN Ia want different ends of this
curve, so the choice of $\beta$ belongs to the scientist and should be reported
with the result.

Remember also the caveat of §2: with 10 TDEs in the test split, each TDE is ten
points of TDE recall.

---
## 8. What transfers to other problems

Nothing here is specific to ZTF. The reasoning generalises:

1. **Count the rare class before choosing a method.** Tens of examples means
   features; tens of thousands means representation learning. This one number
   should drive the decision more than fashion does.
2. **Spend the effort on the representation.** §2 and §3 together: moving along
   the feature axis changed the score far more than swapping the learner, and
   catalogue context is part of the representation.
3. **Prefer a model we can question.** When the answer is "nuclear, blue, and
   staying blue", a domain expert can check it.
4. **Never report accuracy alone.** §3 had four models within four points of
   accuracy and a factor of five in calibration error; §7 had a knob that more
   than doubled rare-class recall while accuracy moved by about two points. Under
   strong imbalance, accuracy is nearly blind. Report balanced accuracy, macro
   $F_1$, AUPRC and ECE, and show a reliability diagram.
5. **Measure calibration, never assume it from the model family.** Neither "nets
   are overconfident" nor "trees are calibrated" survived §5.
6. **Do not threshold on confidence: it deletes the rare classes.** Use
   prediction sets, calibrate them on data nothing else has touched, and check
   the realised coverage *for each class*.
7. **Make the imbalance trade-off explicit** and hand the choice to the scientist.

And the honest caveat: the numbers in this notebook come from a single
train/validation/test split, with 10 TDEs and 24 SLSNe in the test set, so small
differences here are noise. The published FLARE results use five-fold
cross-validation over all 10,497 benchmark objects. If we take one methodological
habit away, it should be that one.

---
## 9. Everything we just saw is on GitHub

The whole package (features, models, conformal layer, anomaly layer, pretrained
weights, the benchmark and this notebook) is one repository:

### https://github.com/applecider-ml/flare

```bash
git clone https://github.com/applecider-ml/flare.git
cd flare
pip install -e .
```

**Classify objects with the pretrained six-class model** (no training, no GPU):

```python
import flare

clf   = flare.load_classifier()                                # hierarchical, six classes
proba = clf.predict_proba_from_files(["ZTF21abcdxyz.npz"])     # (N, 6)
label = clf.predict_from_files(["ZTF21abcdxyz.npz"])           # ['TDE']
sets  = clf.prediction_sets_from_files(["ZTF21abcdxyz.npz"])   # [['TDE', 'AGN']]
```

From the command line:

```bash
flare predict ZTF19acbzgog                    # JSON for one object
flare report ZTF19acbzgog -o report.html      # a self-contained report page
```

**Rerun this notebook** on the benchmark that ships with the repository:

```bash
python -c "from flare import benchmark as B; B.prepare()"   # unpacks the light curves
jupyter lab notebooks/why_flare_works.ipynb
```

**Retrain the shipped models** with the scripts in `scripts/`: start with
`python scripts/setup_data.py`; the shipped models come from
`python scripts/train_bts6.py`.

Other notebooks in the repository:

| Notebook | What it is |
|---|---|
| `flare_tutorial.ipynb` | hands-on introduction: light curves → features → trees → conformal |
| `similarity_search.ipynb` | "find me more like this one" in the FLARE feature space |
| `rubin_flare_lesson.ipynb` | FLARE on simulated Rubin transients in six filters |
| `ztf_summer_school_v2/deep_architecture.ipynb` | a deep sequence classifier on raw light curves, built from scratch |
| `ztf_summer_school_v2/flare_pipeline.ipynb` | the pretrained pipeline end to end: classification, conformal sets, anomaly detection |

Questions, issues and pull requests are welcome.